# 01 — Groundwater Transport (GWT) Model

This notebook adds a MODFLOW 6 **Groundwater Transport (GWT)** model to the simulation
built in `00_gwf.ipynb`. It simulates the migration of **Acid Mine Drainage (AMD)**
leaking from a tailings storage facility (TSF) into the aquifer.

> **Pre-requisite:** Run `00_gwf.ipynb` first so that the GWF model files exist in `model/`.

---

## Where is the TSF?

The TSF is placed **northeast of the pit**, on the upgradient (eastern) side.

With natural flow from east to west, AMD migrates:
1. **Period 1 (dewatering):** the cone of depression accelerates AMD toward the pit;
   dewatering wells partially capture the plume.
2. **Period 2 (recovery):** pumping stops, the plume resumes westward migration past
   the MAR zone and, over decades, reaches the GDE.

## Imports

In [ ]:
import os
import shutil
import flopy
import pyemu
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from herebedragons import specify_tsf_cells, plot_setup, specify_pit_cells

## Transport parameters

These are the only new parameters needed beyond what was defined in `00_gwf.ipynb`.

- **Porosity (θ = 0.3):** effective porosity — the fraction of the aquifer volume
  actually available to flowing water. Determines the pore-water velocity:
  $v = q / \theta$, so lower porosity → faster solute travel.
- **Longitudinal dispersivity (αL = 10 m):** spreading in the flow direction due to
  aquifer heterogeneity (grid-scale macrodispersion). A rule of thumb is αL ≈ 0.1 × travel distance, so 10 m is
  appropriate for a 100 m scale problem.
- **Transverse dispersivity (αT = 1 m):** lateral spreading perpendicular to flow;
  typically 1/10 of αL.
- **TSF concentration (C = 1.0):** normalised AMD tracer. Think of 1.0 as the peak
  concentration leaking from the tailings (e.g. 1000 mg/L sulfate); background = 0.

In [ ]:
# AMD tracer
conc_amd = 1.0    # normalised concentration at TSF source (dimensionless)
conc_bg  = 0.0    # background concentration

# Aquifer transport properties
porosity = 0.30   # effective porosity
alpha_L  = 20.0   # longitudinal dispersivity (m)
alpha_T  = 20.0    # transverse dispersivity (m)

In [ ]:
ws = os.path.join('model', 'flow_transport')

if os.path.exists(ws):
    shutil.rmtree(ws)
os.makedirs(ws)

# copy flow model into directory
shutil.copytree(os.path.join('model', 'flow_only'), ws, dirs_exist_ok=True)

## Load the GWF simulation

We load the simulation written by `00_gwf.ipynb` so the GWT model can be added to the
same `MFSimulation` object. This means both models share the same TDIS (time
discretization) and are solved together in one MODFLOW 6 run.

`pitcells` are re-derived from the `idomain = 2` zone that `specify_pit_cells` tagged
in the GWF model — no need to rebuild the geometry from scratch.

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=ws)
gwf = sim.get_model()

# Re-derive pit cells from the idomain zone tag
pitcells = np.argwhere(gwf.dis.idomain.array[0] == 2)

# read the number of layers, rows and columns from parent flow model
nlay = gwf.dis.nlay.get_data()
nrow = gwf.dis.nrow.get_data()
ncol = gwf.dis.ncol.get_data()

## GWT model object

`ModflowGwt` creates the transport model container inside the existing simulation.
By using the same `sim` object as GWF, MODFLOW 6 will automatically couple them
via the GWF6-GWT6 exchange added at the end of this notebook.

In [ ]:
gwt = flopy.mf6.ModflowGwt(sim, modelname='gwt')

## GWT solver (IMS)

The transport equation is linear (concentration does not affect flow), so a separate,
lighter IMS is registered for GWT. 

In [ ]:
ims_gwt = flopy.mf6.ModflowIms(
    sim,
    pname='ims_gwt',
    complexity='COMPLEX',
    outer_dvclose=1e-6,
    outer_maximum=500,
    linear_acceleration='BICGSTAB',
    filename=f"{gwt.name}.ims"
)
sim.register_ims_package(ims_gwt, [gwt.name])

## Grid (DIS)

GWT requires its own DIS package, but the grid must be identical to GWF's.
We read the arrays directly from the loaded GWF model rather than re-specifying them.

In [ ]:
dis_gwt = flopy.mf6.ModflowGwtdis(
    gwt,
    nlay=nlay, nrow=nrow, ncol=ncol,
    delr=gwf.dis.delr.get_data(),
    delc=gwf.dis.delc.get_data(),
    top=gwf.dis.top.get_data(),
    botm=gwf.dis.botm.get_data(),
    idomain=gwf.dis.idomain.get_data(),
    length_units='meters',
)

## Initial conditions (IC)

All cells start with zero AMD concentration — the aquifer is clean before mining begins.

In [ ]:
ic_gwt = flopy.mf6.ModflowGwtic(gwt, strt=conc_bg)

## Mobile storage and transfer (MST)

MST defines how much solute is stored per unit volume of aquifer. The key parameter
is **porosity** — the fraction of volume occupied by water where solute resides.

Porosity controls the **pore-water velocity**: $v = q/\theta$. With $\theta = 0.3$
and a Darcy flux of $q = K \cdot i = 5 \times 0.005 = 0.025$ m/day, the average
pore-water velocity is ~0.08 m/day → AMD travels ~30 m/year under natural gradient.

In [ ]:
mst = flopy.mf6.ModflowGwtmst(gwt, porosity=porosity)

## Advection (ADV)

ADV moves solute with the groundwater flow field computed by GWF.

`scheme='TVD'` (Total Variation Diminishing) is a higher-order scheme that reduces
numerical oscillations near sharp concentration fronts — important here because the
TSF injects a step-change in concentration. The simpler `'UPSTREAM'` scheme is
faster but produces more numerical diffusion (artificially smears the plume front).

In [ ]:
adv = flopy.mf6.ModflowGwtadv(gwt, scheme='UPSTREAM')

## Dispersion (DSP)

DSP spreads the plume beyond pure advection due to:
- **Mechanical dispersion** — solute travels at different speeds through different pore
  paths (fast through large pores, slow through small ones). Scales with velocity:
  $D_L = \alpha_L \cdot v$, $D_T = \alpha_T \cdot v$.
- **Molecular diffusion** (`diffc`) — Brownian motion of solute molecules. Minor at
  groundwater velocities; set to 0 here for simplicity.

The dispersion tensor gives the plume an elongated shape: longer in the flow direction
(longitudinal) and shorter transversely. With αL/αT = 10:1, the plume will be
approximately 10× longer east–west than it is wide north–south.

In [ ]:
dsp = flopy.mf6.ModflowGwtdsp(
    gwt,
    xt3d_off=True,   # use standard finite-difference dispersion
    diffc=0.0,       # molecular diffusion (m²/day)
    alh=alpha_L,     # longitudinal dispersivity (m)
    ath1=alpha_T,    # transverse dispersivity (m)
)

## Source/sink mixing (SSM)

SSM tells MODFLOW what concentration to assign to water entering the aquifer through
boundary conditions:

- **Sinks** (dewatering wells, drain): remove solute at the local aquifer concentration.
- **Sources** (GHB inflow, recharge, MAR wells): inject water at a specified concentration.

With `sources=[]` all sources inject **clean water at C = 0**, which is physically
reasonable: regional inflow is uncontaminated, rain is clean, and MAR injects treated
water. This means MAR actively **dilutes** the AMD plume — you can test the dilution
effect by comparing runs with and without MAR.

In [ ]:
# get ghb name

sources = [[gwf.ghb.package_name, 'aux', 'tracer'],
           [gwf.wel[0].package_name, 'aux', 'tracer'],
           [gwf.drn.package_name, 'aux', 'tracer'],
           [gwf.rcha.package_name, 'aux', 'tracer']
            ]
ssm = flopy.mf6.ModflowGwtssm(gwt, sources= sources,
                              save_flows=True,
                              print_flows=True,
                              filename=f"{gwt.name}.ssm")

## Tailings source — Constant Concentration (CNC)

CNC fixes the concentration at specified cells to a constant value for the duration
of a stress period. It is the transport equivalent of a constant-head boundary in GWF.

The TSF footprint is a **5 × 3 cell rectangle (100 m × 60 m)** centred northeast of
the pit. The concentration is held at `conc_amd = 1.0` during periods 1 and 2
(throughout mining and into recovery), representing continuous leaching from the
tailings pile.

> **Try it:** set `conc_amd = 0` in period 2 (`stress_period_data={1: tsf_cells}`) to
> simulate a TSF that is capped and stops leaching when mining ends.

In [ ]:
tsf_cells = specify_tsf_cells()
pit_side_length= 200
pitcells    = specify_pit_cells(gwf, pit_side_length)
pitcells

tsf_cells
# cnc = flopy.mf6.ModflowGwtcnc(
#     gwt,
#     stress_period_data={1: tsf_cells, 2: tsf_cells},
# )

cnc = flopy.mf6.ModflowGwtcnc(
    gwt,
    stress_period_data={2: tsf_cells},
)

## Output control (OC)

Save the concentration array to a binary `.ucn` file every time step.
This is read by flopy's `HeadFile` utility (with `text='CONCENTRATION'`) for
post-processing — same interface as reading heads from a `.hds` file.

In [ ]:
oc_gwt = flopy.mf6.ModflowGwtoc(
    gwt,
    concentration_filerecord=f'{gwt.name}.ucn',
    saverecord=[('CONCENTRATION', 'ALL')],
    printrecord=[('BUDGET', 'ALL')],
)

## GWF–GWT exchange

The `GWF6-GWT6` exchange registers the coupling between the two models in the
simulation's name file. At each time step MODFLOW:
1. Solves GWF → computes heads and cell-by-cell flows.
2. Passes those flows to GWT.
3. Solves GWT → computes concentrations driven by the GWF flow field.

This is a **one-way coupling**: flow drives transport, but transport does not feed back
into flow (concentration does not change density or viscosity here). For density-driven
flow (e.g. saltwater intrusion) you would need a fully coupled GWF-GWT solve.

In [ ]:
gwfgwt = flopy.mf6.ModflowGwfgwt(
    sim,
    exgtype='GWF6-GWT6',
    exgmnamea=gwf.name,
    exgmnameb=gwt.name,
    filename=f"{gwt.name}.gwfgwt"
)

## Visualise model setup

Same map as `00_gwf.ipynb` but with the TSF footprint overlaid in blue.
Verify that the TSF sits northeast of the pit, upgradient of all other features.

In [ ]:
plot_setup(gwf, pitcells, tsf_cells=tsf_cells)

## Write and run

Re-writing the simulation updates the GWF name file (adds budget output) and
writes all GWT input files. MODFLOW 6 then solves GWF and GWT together.

In [ ]:
sim.set_all_data_external
sim.write_simulation()


In [ ]:
pyemu.os_utils.run('mf6', cwd=ws)

## Visualise the AMD plume

We read the concentration file and plot snapshots at three key moments:
- Pre-mining
- End of period 2 (dewatering off, t = 10 years) — plume under pumping capture
- End of simulation (t = 110 years) — GDE impact?

The TSF is marked as a black square; the pit outline is shown for reference.

In [ ]:
import matplotlib.patches as patches

ucn_file = os.path.join(ws, f'{gwt.name}.ucn')
cobj     = flopy.utils.HeadFile(ucn_file, text='CONCENTRATION')
kstpkper = np.array(cobj.get_kstpkper())   # [[kstp, kper], ...]
_, idx = np.unique(kstpkper[:, 1], return_index=True)
# np.unique returns first occurrence; we want last, so reverse trick:
idx_last = np.array([
    np.where(kstpkper[:, 1] == kper)[0][-1]
    for kper in np.unique(kstpkper[:, 1])
])

kstpkper_lean = kstpkper[idx_last]

labels   = [f'{t/365:.0f} yr' for t in np.array(cobj.get_times())[idx_last]]

mg        = gwf.modelgrid
idom      = gwf.dis.idomain.array[0]
cell_size = float(gwf.dis.delr.get_data()[0])

# Pit rectangle from cell centres
pit_x = [mg.xcellcenters[r, c] for r, c in pitcells]
pit_y = [mg.ycellcenters[r, c] for r, c in pitcells]
pit_rect_kw = dict(
    xy=(min(pit_x) - cell_size/2, min(pit_y) - cell_size/2),
    width=max(pit_x) - min(pit_x) + cell_size,
    height=max(pit_y) - min(pit_y) + cell_size,
    lw=1.2, edgecolor='k', facecolor='none',
)

# TSF rectangle from cell centres
tsf_ids = [c[0] for c in specify_tsf_cells()]
tsf_x   = [mg.xcellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_y   = [mg.ycellcenters[r, c] for (_, r, c) in tsf_ids]
tsf_rect_kw = dict(
    xy=(min(tsf_x) - cell_size/2, min(tsf_y) - cell_size/2),
    width=max(tsf_x) - min(tsf_x) + cell_size,
    height=max(tsf_y) - min(tsf_y) + cell_size,
    lw=1.5, edgecolor='navy', facecolor='steelblue', alpha=0.4,
)

cmap = mcolors.LinearSegmentedColormap.from_list('amd', ['white', 'gold', 'darkorange', 'firebrick'])
fig, axes = plt.subplots(1, len(kstpkper_lean), figsize=(14, 4), constrained_layout=True)

for ax, ksp, lbl in zip(axes, kstpkper_lean, labels):
    data = cobj.get_data(kstpkper=tuple(ksp))[0].astype(float)
    data[idom <= 0] = np.nan

    mapview = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax)
    pc = mapview.plot_array(data,  cmap=cmap, alpha=0.85)
    mapview.plot_grid(alpha=0.05)
    mapview.plot_bc("wel-mar",  kper=1)
    ax.add_patch(patches.Rectangle(**pit_rect_kw))
    ax.add_patch(patches.Rectangle(**tsf_rect_kw))
    ax.set_title(lbl)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(pc, ax=list(axes), label='AMD tracer (normalised)', shrink=0.7)
plt.show()